# Initial Value Problem Solvers

This notebook demonstrates `torchscience.integration.initial_value_problem` - differentiable ODE solvers for PyTorch.

We'll explore three physics examples:
1. **Double Pendulum** - chaotic dynamics with adaptive stepping
2. **Lotka-Volterra** - ecology model with TensorDict state
3. **Neural ODE** - learning dynamics with adjoint gradients

Each example showcases different solver capabilities with interactive Plotly visualizations and Manim animations.

In [12]:
import time

import numpy as np
import plotly.express as px  # noqa: F401
import plotly.graph_objects as go  # noqa: F401
import torch  # noqa: F401
from plotly.subplots import make_subplots  # noqa: F401
from tensordict import TensorDict  # noqa: F401

try:
    from manim import *  # noqa: F401, F403

    MANIM_AVAILABLE = True
except ImportError:
    MANIM_AVAILABLE = False
    print("Note: manim not installed. Animation examples will be skipped.")
from IPython.display import Video  # noqa: F401

from torchscience.integration.initial_value_problem import (  # noqa: F401
    adjoint,
    backward_euler,
    dormand_prince_5,
    euler,
    midpoint,
    runge_kutta_4,
)

In [13]:
# Configuration
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Plotly defaults for dark theme
plotly_template = "plotly_dark"  # noqa: F841

Using device: cpu


## 1. Double Pendulum: Chaos and Adaptive Stepping

The double pendulum is a classic chaotic system - two rigid rods connected at joints, swinging under gravity. Small changes in initial conditions lead to dramatically different trajectories.

**State vector:** `[theta1, theta2, omega1, omega2]`
- `theta1`, `theta2`: angles from vertical
- `omega1`, `omega2`: angular velocities

**Why it's a good test case:**
- Chaotic dynamics require adaptive step sizes
- Large state changes stress error estimation
- Visually dramatic for demonstrations

In [14]:
def double_pendulum_dynamics(t, state):
    """
    Double pendulum equations of motion.

    Parameters:
        t: time (unused, system is autonomous)
        state: [theta1, theta2, omega1, omega2]

    Returns:
        [dtheta1/dt, dtheta2/dt, domega1/dt, domega2/dt]
    """
    # Physical parameters
    g = 9.81  # gravity
    L1 = 1.0  # length of rod 1
    L2 = 1.0  # length of rod 2
    m1 = 1.0  # mass 1
    m2 = 1.0  # mass 2

    theta1, theta2, omega1, omega2 = (
        state[..., 0],
        state[..., 1],
        state[..., 2],
        state[..., 3],
    )

    delta = theta2 - theta1

    # Denominator terms
    den1 = (m1 + m2) * L1 - m2 * L1 * torch.cos(delta) ** 2
    den2 = (L2 / L1) * den1

    # Angular accelerations
    domega1 = (
        -m2 * L1 * omega1**2 * torch.sin(delta) * torch.cos(delta)
        + m2 * g * torch.sin(theta2) * torch.cos(delta)
        + m2 * L2 * omega2**2 * torch.sin(delta)
        - (m1 + m2) * g * torch.sin(theta1)
    ) / den1

    domega2 = (
        +m2 * L2 * omega2**2 * torch.sin(delta) * torch.cos(delta)
        + (m1 + m2) * g * torch.sin(theta1) * torch.cos(delta)
        + (m1 + m2) * L1 * omega1**2 * torch.sin(delta)
        - (m1 + m2) * g * torch.sin(theta2)
    ) / den2

    return torch.stack([omega1, omega2, domega1, domega2], dim=-1)

In [ ]:
# Initial conditions: starting nearly inverted for dramatic chaos
# [theta1, theta2, omega1, omega2]
y0 = torch.tensor([3.0, 3.0, 0.0, 0.0], dtype=torch.float64, device=device)

# Solve with dormand_prince_5
start = time.perf_counter()
y_final, interp = dormand_prince_5(
    double_pendulum_dynamics,
    y0,
    t_span=(0.0, 20.0),
    rtol=1e-8,
    atol=1e-10,
    max_steps=50000,
)
elapsed = time.perf_counter() - start

print(f"Integration completed in {elapsed:.3f}s")
print(f"Number of steps: {interp.n_steps}")
print(f"Final state: theta1={y_final[0]:.3f}, theta2={y_final[1]:.3f}")

In [ ]:
# Sample dense trajectory using interpolant
t_dense = torch.linspace(0, 20, 2000, dtype=torch.float64, device=device)
trajectory = interp(t_dense)

theta1 = trajectory[:, 0].cpu().numpy()
theta2 = trajectory[:, 1].cpu().numpy()
omega1 = trajectory[:, 2].cpu().numpy()
omega2 = trajectory[:, 3].cpu().numpy()
t_np = t_dense.cpu().numpy()

In [ ]:
# 3D trajectory: (theta1, theta2, t) colored by omega1
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=theta1,
            y=theta2,
            z=t_np,
            mode="lines",
            line=dict(
                color=omega1,
                colorscale="Viridis",
                width=2,
                colorbar=dict(title="omega1"),
            ),
            name="Trajectory",
        )
    ]
)

fig.update_layout(
    title="Double Pendulum: Chaotic Trajectory",
    scene=dict(
        xaxis_title="theta1 (rad)",
        yaxis_title="theta2 (rad)",
        zaxis_title="time (s)",
    ),
    template=plotly_template,
    width=800,
    height=600,
)
fig.show()

In [ ]:
# Phase portrait: (theta1, omega1) colored by time
fig = go.Figure(
    data=[
        go.Scatter(
            x=theta1,
            y=omega1,
            mode="lines",
            line=dict(
                color=t_np,
                colorscale="Plasma",
                width=1,
                colorbar=dict(title="time (s)"),
            ),
        )
    ]
)

fig.update_layout(
    title="Phase Portrait: theta1 vs omega1",
    xaxis_title="theta1 (rad)",
    yaxis_title="omega1 (rad/s)",
    template=plotly_template,
    width=700,
    height=500,
)
fig.show()

In [ ]:
# Convert angles to Cartesian coordinates
L1, L2 = 1.0, 1.0

x1 = L1 * np.sin(theta1)
y1 = -L1 * np.cos(theta1)
x2 = x1 + L2 * np.sin(theta2)
y2 = y1 - L2 * np.cos(theta2)

print(
    f"Mass 1 range: x=[{x1.min():.2f}, {x1.max():.2f}], y=[{y1.min():.2f}, {y1.max():.2f}]"
)
print(
    f"Mass 2 range: x=[{x2.min():.2f}, {x2.max():.2f}], y=[{y2.min():.2f}, {y2.max():.2f}]"
)

In [ ]:
# Double pendulum animation (requires manim)
if MANIM_AVAILABLE:
    from manim import (
        BLUE,
        ORIGIN,
        RED,
        WHITE,
        YELLOW,
        Dot,
        Line,
        Scene,
        TracedPath,
    )

    class DoublePendulumScene(Scene):
        def construct(self):
            # Physical parameters
            L1, L2 = 2.0, 2.0  # Scaled for visibility

            # Get trajectory data (use global variables from previous cells)
            # Sample every 10th point for smoother animation
            theta1_anim = theta1[::10]
            theta2_anim = theta2[::10]

            # Create pendulum components
            pivot = Dot(ORIGIN, color=WHITE)

            def get_positions(i):
                t1, t2 = theta1_anim[i], theta2_anim[i]
                p1 = np.array([L1 * np.sin(t1), -L1 * np.cos(t1), 0])
                p2 = p1 + np.array([L2 * np.sin(t2), -L2 * np.cos(t2), 0])
                return p1, p2

            p1_0, p2_0 = get_positions(0)

            rod1 = Line(ORIGIN, p1_0, color=BLUE, stroke_width=4)
            rod2 = Line(p1_0, p2_0, color=RED, stroke_width=4)
            mass1 = Dot(p1_0, color=BLUE, radius=0.15)
            mass2 = Dot(p2_0, color=RED, radius=0.15)

            # Trail for mass2
            trail = TracedPath(
                mass2.get_center,
                stroke_color=YELLOW,
                stroke_width=1,
                stroke_opacity=0.5,
            )

            self.add(pivot, rod1, rod2, mass1, mass2, trail)

            # This defines the scene - to render, use: manim -pql notebook.py DoublePendulumScene
            # Or in a Jupyter cell with manim kernel: %%manim -qm DoublePendulumScene
            self.wait(2)
else:
    print("Manim not available - skipping animation cell")

### Observations

- **Adaptive Stepping:** Notice how `dormand_prince_5` takes many small steps during rapid changes and larger steps during calmer periods.
- **Step Count:** The solver efficiently handles the chaotic dynamics with automatic step size adaptation.
- **Sensitivity:** Try changing initial conditions by 0.01 rad - the trajectories diverge dramatically within seconds.
- **Dense Output:** The interpolant provides smooth trajectories at any time point without re-solving.

## 2. Lotka-Volterra: TensorDict and Solver Comparison

The Lotka-Volterra equations model predator-prey dynamics in ecology. Populations oscillate in closed orbits - when prey is abundant, predators thrive; as prey declines, predators starve.

**Equations:**
- `dx/dt = alpha*x - beta*x*y` (prey growth minus predation)
- `dy/dt = delta*x*y - gamma*y` (predator growth minus death)

**Why it's a good test case:**
- Simple, intuitive dynamics
- Closed orbits test energy conservation
- TensorDict state demonstrates structured data handling
- Smooth enough for fixed-step solvers

In [ ]:
def lotka_volterra(t, state):
    """
    Lotka-Volterra predator-prey dynamics using TensorDict.

    Parameters:
        t: time (unused, system is autonomous)
        state: TensorDict with 'prey' and 'predator' keys

    Returns:
        TensorDict with derivatives
    """
    # Parameters
    alpha = 1.0  # prey growth rate
    beta = 0.1  # predation rate
    delta = 0.075  # predator growth rate
    gamma = 1.5  # predator death rate

    x = state["prey"]  # prey population
    y = state["predator"]  # predator population

    dx = alpha * x - beta * x * y
    dy = delta * x * y - gamma * y

    return TensorDict(
        {
            "prey": dx,
            "predator": dy,
        },
        batch_size=state.batch_size,
    )

In [ ]:
# Initial populations
state0 = TensorDict(
    {
        "prey": torch.tensor([40.0], device=device),
        "predator": torch.tensor([9.0], device=device),
    },
    batch_size=[],
)

# Solve with runge_kutta_4 (fixed step, good for smooth problems)
state_final, interp_lv = runge_kutta_4(
    lotka_volterra,
    state0,
    t_span=(0.0, 50.0),
    dt=0.01,
)

print(
    f"Initial: prey={state0['prey'].item():.1f}, predator={state0['predator'].item():.1f}"
)
print(
    f"Final:   prey={state_final['prey'].item():.1f}, predator={state_final['predator'].item():.1f}"
)

In [ ]:
# Compare solvers on Lotka-Volterra
solvers = {
    "euler": (euler, {"dt": 0.01}),
    "midpoint": (midpoint, {"dt": 0.05}),
    "runge_kutta_4": (runge_kutta_4, {"dt": 0.1}),
    "dormand_prince_5": (dormand_prince_5, {}),
}

# Reference solution with tight tolerances
state_ref, _ = dormand_prince_5(
    lotka_volterra, state0, t_span=(0.0, 50.0), rtol=1e-10, atol=1e-12
)

results = {}
interpolants = {}  # Store interpolants for cell 16
for name, (solver, kwargs) in solvers.items():
    start = time.perf_counter()
    state_final, interp = solver(
        lotka_volterra, state0, t_span=(0.0, 50.0), **kwargs
    )
    elapsed = time.perf_counter() - start

    error = torch.sqrt(
        (state_final["prey"] - state_ref["prey"]) ** 2
        + (state_final["predator"] - state_ref["predator"]) ** 2
    ).item()

    results[name] = {
        "time": elapsed,
        "error": error,
        "prey_final": state_final["prey"].item(),
        "predator_final": state_final["predator"].item(),
    }
    interpolants[name] = interp  # Store interpolant

# Display comparison
import pandas as pd

df = pd.DataFrame(results).T
df.columns = ["Time (s)", "Error", "Prey Final", "Predator Final"]
df

In [ ]:
# Time series comparison
t_eval = torch.linspace(0, 50, 500, device=device)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=("Prey Population", "Predator Population"),
)

colors = {
    "euler": "red",
    "midpoint": "orange",
    "runge_kutta_4": "green",
    "dormand_prince_5": "blue",
}

for name in solvers.keys():
    interp = interpolants[name]  # Reuse stored interpolant

    # Batched query - much faster
    states = interp(t_eval)
    # Handle both TensorDict and tensor return types
    if hasattr(states, "keys"):  # TensorDict
        prey_traj = states["prey"].squeeze().cpu().numpy()
        pred_traj = states["predator"].squeeze().cpu().numpy()
    else:
        prey_traj = states[..., 0].cpu().numpy()
        pred_traj = states[..., 1].cpu().numpy()

    fig.add_trace(
        go.Scatter(
            x=t_eval.cpu().numpy(),
            y=prey_traj,
            name=name,
            line=dict(color=colors[name]),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=t_eval.cpu().numpy(),
            y=pred_traj,
            name=name,
            line=dict(color=colors[name]),
            showlegend=False,
        ),
        row=2,
        col=1,
    )

fig.update_layout(
    title="Solver Comparison: Lotka-Volterra",
    template=plotly_template,
    height=600,
)
fig.update_xaxes(title_text="Time", row=2, col=1)
fig.update_yaxes(title_text="Prey", row=1, col=1)
fig.update_yaxes(title_text="Predator", row=2, col=1)
fig.show()